In [ ]:
import numpy as np
import sys
import os

# Add the project root to the path
sys.path.append('/home/justin/code/point-to-pose')

from point2pose.modules.register.svd_register import SVDRegister
from point2pose.modules.register.svd_residual_outlier import SVDResidualOutlierRegister
from point2pose.utils.transform import transform_pts

# ---- 1) Load the whole npz ----
D = np.load('/home/justin/code/point-to-pose/debug/pipeline/meta_data/meata_data.npz', allow_pickle=True)  # dict-like

print("Keys:", list(D.files))  # discover what's inside
N = len(D["frame_id"])        # number of rows/frames
print("Num rows:", N)

# ---- 2) Helper to unpack ragged fields ----
def unpack_ragged(name: str, store: dict):
    data    = store[f"{name}_data"]
    offsets = store[f"{name}_offsets"]
    lengths = store[f"{name}_lengths"]
    out = []
    for off, L in zip(offsets, lengths):
        flat_data = data[off:off+L]
        # Reshape to (N, 3) assuming 3D points
        if len(flat_data) % 3 == 0:
            reshaped_data = flat_data.reshape(-1, 3)
        else:
            print(f"Warning: {name} data length {len(flat_data)} not divisible by 3")
            reshaped_data = flat_data  # Keep as 1D if can't reshape
        out.append(reshaped_data)
    return out  # -> list of (N, 3) ndarrays (one per row)

# ---- 3) Access fixed-shape fields (already stacked) ----
timestamp  = D["timestamp"]          # shape (N,)
frame_id   = D["frame_id"]           # shape (N,)

print(D["reg_key_points_data"].shape)

# ---- 4) Access ragged fields ----
reg_key_points_list = unpack_ragged("reg_key_points", D)  # list of (Mi,3) float arrays
reg_cur3d_list = unpack_ragged("reg_curr3d", D)            # list of (Mi,3) float arrays

print(f"\nExtracted registration data:")
print(f"  reg_key_points_list: {len(reg_key_points_list)} frames")
print(f"  reg_cur3d_list: {len(reg_cur3d_list)} frames")

# Show shapes for first few frames
# for i in range(min(3, len(reg_key_points_list))):
#     print(f"  Frame {i}: reg_key_points {reg_key_points_list[i].shape}, reg_cur3d {reg_cur3d_list[i].shape}")
#     print(reg_key_points_list[i])


# Create a register instance for debugging
config = {
    'debug_level': 1,
    'debug_dir': '/home/justin/code/point-to-pose/debug/register_test'
}
svd_register = SVDResidualOutlierRegister(config)

print(f"\nCreated SVDRegister instance with debug level: {config['debug_level']}")

# Store the data for use in other cells
print(f"\nData loaded successfully! Available variables:")
print(f"  - D: Full data dictionary")
print(f"  - N: Number of frames ({N})")
print(f"  - frame_id, obj_id, res_mean, num_points: Fixed-shape arrays")
print(f"  - reg_key_points_list, reg_cur3d_list: Lists of point clouds")
print(f"  - register: SVDResidualOutlierRegister instance")


In [ ]:
import numpy as np
import plotly.graph_objects as go

def _apply_tf(points: np.ndarray, tf: np.ndarray) -> np.ndarray:
    """Apply a 4x4 homogeneous transform to Nx3 points."""
    points = np.asarray(points)
    assert points.ndim == 2 and points.shape[1] == 3, "points must be (N,3)"
    assert tf.shape == (4, 4), "tf must be a 4x4 homogeneous matrix"
    homog = np.c_[points, np.ones(len(points))]
    out = homog @ tf.T
    return out[:, :3]

def vis_reg_points(
    src: np.ndarray,
    trg: np.ndarray,
    tf: np.ndarray,
    backend: str = "plotly",
    point_size: float = 2.0,
    src_color=None,            # None, single color string (e.g. 'red'), or per-point Nx3 in [0,1] or 0..255
    trg_color=None,
    title: str = "Registered point clouds (src→trg)"
):
    """
    Visualize two point clouds after applying tf to src.
    Args:
        src, trg: (N,3) and (M,3) float arrays.
        tf: (4,4) homogeneous transform that maps src -> trg frame.
        backend: 'plotly' (inline, easy) or 'k3d' (very fast for huge clouds).
        point_size: marker size (Plotly) or glyph size (k3d).
        src_color, trg_color: None, single color string/int, or per-point Nx3.
    """
    src_t = _apply_tf(src, tf)

    if backend.lower() == "plotly":
        import plotly.graph_objects as go

        def _to_plotly_color(arr, fallback):
            if arr is None:
                return fallback
            arr = np.asarray(arr)
            if arr.ndim == 1 and arr.size == 3:  # single RGB triplet
                arr = np.tile(arr, (1,1))
            if arr.ndim == 2 and arr.shape[1] == 3:
                # normalize if in 0..1
                if arr.max() <= 1.0:
                    arr = (arr * 255).astype(np.uint8)
                else:
                    arr = arr.astype(np.uint8)
                return [f"rgb({r},{g},{b})" for r, g, b in arr]
            # otherwise assume a CSS color string or list Plotly can handle
            return arr

        src_col = _to_plotly_color(src_color, "red")
        trg_col = _to_plotly_color(trg_color, "royalblue")

        fig = go.Figure()
        fig.add_trace(go.Scatter3d(
            x=trg[:,0], y=trg[:,1], z=trg[:,2],
            mode="markers",
            marker=dict(size=point_size, opacity=0.85, color=trg_col),
            name="target (trg)"
        ))
        fig.add_trace(go.Scatter3d(
            x=src_t[:,0], y=src_t[:,1], z=src_t[:,2],
            mode="markers",
            marker=dict(size=point_size, opacity=0.85, color=src_col),
            name="source transformed (src·tf)"
        ))
        fig.update_layout(
            title=title,
            scene=dict(aspectmode="data", xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
            margin=dict(l=0, r=0, t=40, b=0),
            width=900, height=700,
            legend=dict(itemsizing="constant")
        )
        return fig.show()

    elif backend.lower() == "k3d":
        import k3d
        from IPython.display import display

        def _to_k3d_colors(arr, n, fallback_hex):
            if arr is None:
                return np.full(n, fallback_hex, dtype=np.uint32)
            arr = np.asarray(arr)
            if arr.ndim == 1 and arr.size == 3:
                arr = np.tile(arr, (n,1))
            if arr.ndim == 2 and arr.shape[1] == 3:
                if arr.max() <= 1.0:
                    arr = (arr * 255).astype(np.uint8)
                arr = arr.astype(np.uint8)
                return ((arr[:,0].astype(np.uint32) << 16) |
                        (arr[:,1].astype(np.uint32) << 8) |
                         arr[:,2].astype(np.uint32))
            # single packed int color or array of packed ints
            return arr.astype(np.uint32)

        trg_pts = trg.astype(np.float32)
        src_pts = src_t.astype(np.float32)
        trg_cols = _to_k3d_colors(trg_color, len(trg_pts), 0x4169E1)  # royalblue
        src_cols = _to_k3d_colors(src_color, len(src_pts), 0xFF0000)  # red

        plot = k3d.plot(grid_visible=False, height=700)
        plot += k3d.points(trg_pts, colors=trg_cols, point_size=point_size*0.01, shader='3d', name='trg')
        plot += k3d.points(src_pts, colors=src_cols, point_size=point_size*0.01, shader='3d', name='src·tf')
        plot.camera_auto_fit = True
        display(plot)
    else:
        raise ValueError("backend must be 'plotly' or 'k3d'")

def _auto_palette(n: int):
    # deterministic-ish palette in [0..255]
    if n <= 0:
        return np.zeros((0, 3), dtype=np.uint8)
    # golden ratio trick in HSV
    h = (np.arange(n) * 0.61803398875) % 1.0
    s = np.full(n, 0.65)
    v = np.full(n, 0.95)
    # hsv -> rgb
    i = np.floor(h * 6).astype(int)
    f = h * 6 - i
    p = v * (1 - s)
    q = v * (1 - f * s)
    t = v * (1 - (1 - f) * s)
    rgb = np.zeros((n, 3))
    idx = (i % 6 == 0); rgb[idx] = np.stack([v[idx], t[idx], p[idx]], 1)
    idx = (i % 6 == 1); rgb[idx] = np.stack([q[idx], v[idx], p[idx]], 1)
    idx = (i % 6 == 2); rgb[idx] = np.stack([p[idx], v[idx], t[idx]], 1)
    idx = (i % 6 == 3); rgb[idx] = np.stack([p[idx], q[idx], v[idx]], 1)
    idx = (i % 6 == 4); rgb[idx] = np.stack([t[idx], p[idx], v[idx]], 1)
    idx = (i % 6 == 5); rgb[idx] = np.stack([v[idx], p[idx], q[idx]], 1)
    return (rgb * 255).astype(np.uint8)

def _normalize_colors(c, n):
    """
    Accept None, CSS string, single RGB (3,), or Nx3 in [0..1] or [0..255].
    Return:
      - for plotly: list of "rgb(r,g,b)" strings or a single CSS string
      - for k3d: packed uint32 per-point
    """
    if c is None:
        return None
    c = np.asarray(c)
    if c.ndim == 1 and c.size == 3:
        c = np.tile(c[None, :], (n, 1))
    if c.ndim == 2 and c.shape[1] == 3:
        if c.max() <= 1.0:
            c = (c * 255).astype(np.uint8)
        else:
            c = c.astype(np.uint8)
        return c
    # let plotly handle strings/lists of strings; k3d path handles ints
    return c

def _plotly_colorize(c_uint8_or_str, n, fallback):
    if c_uint8_or_str is None:
        return fallback
    if isinstance(c_uint8_or_str, np.ndarray) and c_uint8_or_str.ndim == 2:
        return [f"rgb({r},{g},{b})" for r, g, b in c_uint8_or_str]
    return c_uint8_or_str  # string or list of strings

def _k3d_pack_rgb(c_uint8, n, fallback_hex):
    if c_uint8 is None:
        return np.full(n, fallback_hex, dtype=np.uint32)
    if isinstance(c_uint8, np.ndarray) and c_uint8.ndim == 2:
        c = c_uint8.astype(np.uint32)
        return (c[:,0] << 16) | (c[:,1] << 8) | c[:,2]
    # already an int or array of ints
    return np.asarray(c_uint8, dtype=np.uint32)

# ---------- 1) Single cloud ----------
def vis_point_cloud(
    pts: np.ndarray,
    backend: str = "plotly",
    point_size: float = 2.0,
    color=None,                 # None / str / (3,) / Nx3 in [0..1] or [0..255]
    title: str = "Point Cloud",
):
    assert pts.ndim == 2 and pts.shape[1] == 3
    n = pts.shape[0]
    c = _normalize_colors(color, n)

    if backend.lower() == "plotly":
        import plotly.graph_objects as go
        col = _plotly_colorize(c, n, "royalblue")
        fig = go.Figure()
        fig.add_trace(go.Scatter3d(
            x=pts[:,0], y=pts[:,1], z=pts[:,2],
            mode="markers",
            marker=dict(size=point_size, opacity=0.9, color=col),
            name="cloud"
        ))
        fig.update_layout(
            title=title,
            scene=dict(aspectmode="data", xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
            margin=dict(l=0, r=0, t=40, b=0),
            width=900, height=700,
        )
        return fig.show()

    elif backend.lower() == "k3d":
        import k3d
        from IPython.display import display
        cols = _k3d_pack_rgb(c, n, 0x4169E1)  # royalblue
        plot = k3d.plot(grid_visible=False, height=700)
        plot += k3d.points(pts.astype(np.float32), colors=cols, point_size=point_size*0.01, shader='3d')
        plot.camera_auto_fit = True
        return display(plot)

    else:
        raise ValueError("backend must be 'plotly' or 'k3d'")

# ---------- 2) Two clouds (known correspondence, same color per pair) ----------
def vis_corresponded_clouds(
    src: np.ndarray,            # (N,3)
    trg: np.ndarray,            # (N,3) — same order = correspondence
    tf: np.ndarray | None = None,  # optional 4x4 to apply to src before draw
    backend: str = "plotly",
    point_size: float = 2.0,
    palette=None,               # None -> auto palette Nx3; or Nx3 custom colors; or str ignored (auto)
    title: str = "Corresponded clouds (same color = same point)"
):
    assert src.shape == trg.shape and src.shape[1] == 3, "src/trg must be (N,3) with same N"
    N = src.shape[0]
    src_t = _apply_tf(src, tf)

    # build per-point color palette
    if palette is None or (isinstance(palette, str)):
        cols = _auto_palette(N)                     # Nx3 uint8
    else:
        cols = _normalize_colors(palette, N)        # Nx3 uint8
        if not (isinstance(cols, np.ndarray) and cols.ndim == 2 and cols.shape[1] == 3):
            cols = _auto_palette(N)

    if backend.lower() == "plotly":
        import plotly.graph_objects as go
        col_list = [f"rgb({r},{g},{b})" for r, g, b in cols]

        fig = go.Figure()
        # target
        fig.add_trace(go.Scatter3d(
            x=trg[:,0], y=trg[:,1], z=trg[:,2],
            mode="markers",
            marker=dict(size=point_size, opacity=0.9, color=col_list),
            name="trg (corresponded)"
        ))
        # source (transformed)
        fig.add_trace(go.Scatter3d(
            x=src_t[:,0], y=src_t[:,1], z=src_t[:,2],
            mode="markers",
            marker=dict(size=point_size, opacity=0.9, color=col_list),
            name="src (corresponded)"
        ))

        # (optional) tiny lines between corresponded points
        seg_x = np.vstack([src_t[:,0], trg[:,0], np.full(N, np.nan)]).T.reshape(-1)
        seg_y = np.vstack([src_t[:,1], trg[:,1], np.full(N, np.nan)]).T.reshape(-1)
        seg_z = np.vstack([src_t[:,2], trg[:,2], np.full(N, np.nan)]).T.reshape(-1)
        fig.add_trace(go.Scatter3d(
            x=seg_x, y=seg_y, z=seg_z,
            mode="lines",
            line=dict(width=2),
            name="correspondences",
            showlegend=True
        ))

        fig.update_layout(
            title=title,
            scene=dict(aspectmode="data", xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
            margin=dict(l=0, r=0, t=40, b=0),
            width=900, height=700,
            legend=dict(itemsizing="constant"),
        )
        return fig.show()

    elif backend.lower() == "k3d":
        import k3d
        from IPython.display import display
        cols_packed = (cols[:,0].astype(np.uint32) << 16) | (cols[:,1].astype(np.uint32) << 8) | cols[:,2].astype(np.uint32)

        plot = k3d.plot(grid_visible=False, height=700)
        plot += k3d.points(trg.astype(np.float32), colors=cols_packed, point_size=point_size*0.01, shader='3d', name='trg')
        plot += k3d.points(src_t.astype(np.float32), colors=cols_packed, point_size=point_size*0.01, shader='3d', name='src')

        # (optional) correspondence segments
        segs = np.stack([src_t, trg], axis=1).astype(np.float32)  # (N,2,3)
        plot += k3d.lines(positions=segs.reshape(-1, 2, 3), colors=cols_packed, width=0.0015)
        plot.camera_auto_fit = True
        return display(plot)

    else:
        raise ValueError("backend must be 'plotly' or 'k3d'")

In [ ]:
frame_id = 477
threshold_method = "mad"
inlier_thres = 0.05
thres_reduce_factor=0.01
mad_scale = 2.5
min_inliers = 3
max_iter = 5

src_pcd = reg_key_points_list[frame_id]
tgt_pcd = reg_cur3d_list[frame_id]


stats = {}
# number of points
N = src_pcd.shape[0]

# transform the points if the initial pose is given
p0 = src_pcd.copy()

# fit transformation using svd
T = svd_register._svd_fit(p0, tgt_pcd)

inliers = np.ones(N, dtype=bool)

for it in range(max_iter):
    p_T = transform_pts(T, p0)
    residuals = np.linalg.norm(p_T - tgt_pcd, axis=1)

    # choose threshold
    if threshold_method == "mad":
        med = np.median(residuals)
        mad = np.median(np.abs(residuals - med)) + 1e-12
        thr = (
            med + mad_scale * 1.4826 * mad
        )  # 1.4826 makes MAD ~ std for Gaussian
    elif threshold_method == "fixed":
        thr = float(inlier_thres)
    elif threshold_method == "reduce":
        thr = float(inlier_thres) - float(thres_reduce_factor * it)
        if thr < 0:
            thr = 0.001
    else:
        raise ValueError(f"Invalid threshold method: {threshold_method}")

    new_inliers = residuals <= thr

    
    print(f"[Register] iter {it} inliers: {new_inliers.sum()}")
    print(f"[Register] iter {it} thr: {thr}")
    print(f"[Register] iter {it} residuals: {residuals}")
    print(f"[Register] iter {it} res_median: {np.median(residuals)}")
    print(f"[Register] iter {it} res_mean: {np.mean(residuals)}")
    print(f"[Register] iter {it} res_max: {np.max(residuals)}")
    print(f"[Register] iter {it} num_inliers: {new_inliers.sum()}")
    
    vis_corresponded_clouds(src_pcd, tgt_pcd, T)

    # stop if no change or too few inliers
    if (
        np.array_equal(new_inliers, inliers)
        or new_inliers.sum() < min_inliers
        or it == max_iter - 1
    ):
        stats["iter"] = it
        stats["thr"] = thr
        stats["residuals"] = residuals
        stats["inliers"] = inliers

        print(f"[Register] iter {it} res_mean: {np.mean(residuals)}")

        break

    inliers = new_inliers

    # refit on inliers
    T = svd_register._svd_fit(p0[inliers], tgt_pcd[inliers])

    


In [ ]:
import cupoch as cph


criteria = cph.registration.ICPConvergenceCriteria()
criteria.max_iteration = 5

key_points_pcd = cph.geometry.PointCloud()
key_points_pcd.points = cph.utility.Vector3fVector(reg_key_points_list[frame_id].astype(np.float32))

masked_pcd = cph.geometry.PointCloud()
masked_pcd.points = cph.utility.Vector3fVector(reg_cur3d_list[frame_id].astype(np.float32))

refine_result = cph.registration.registration_icp(
        key_points_pcd,
        masked_pcd,
        max_correspondence_distance=0.02,
        init=T.astype(np.float32),
        estimation_method=cph.registration.TransformationEstimationPointToPoint(),
        criteria=criteria,
    )

print(refine_result)

vis_corresponded_clouds(reg_key_points_list[frame_id], reg_cur3d_list[frame_id], refine_result.transformation)
